In [2]:
import pyarrow.dataset as ds

def iter_smiles(path: str, column: str = "SMILES"):
    """
    Yield SMILES strings lazily from a Parquet *directory* or *file*.
    """
    dataset = ds.dataset(path, format="parquet")      # auto-detect shards or file
    for rb in dataset.scanner(columns=[column]).to_batches():
        for s in rb.column(column).to_pylist():
            yield s

In [7]:
# train_trie.py
# Build a TTG-guided token-replacement trie for the PubChem-100 K slice.

import time

import trie_funcs as tf

#from experiments.utils import iter_smiles

SLICE = "chebi_smiles.parquet"
OUT_DIR = "ttg_vocab"            

In [8]:
def compute_trie_metrics(smiles_list, tokenizer):
    """
    Compute all Trie metrics (fertility, mean, variance, normalized entropy) in a single pass.
    
    Args:
        smiles_list: Path to parquet file containing SMILES strings
        tokenizer: Trie tokenizer state with compress_and_len function
        
    Returns:
        tuple: (fertility, mean, variance, normalized_entropy)
    """
    from utils import iter_smiles
    from collections import Counter
    import tqdm
    import math
    
    # Import trie_funcs locally to avoid import issues
    try:
        import trie_funcs as tf
    except ImportError:
        raise ImportError("trie_funcs module is required for Trie metrics")
    
    total_tokens = 0
    total_chars = 0
    token_counts = []
    token_freq = Counter()
    
    # Single pass through the data
    for smi in tqdm.tqdm(iter_smiles(smiles_list), desc="Trie metrics"):
        # Tokenize once per SMILES
        base = tf.tokenize(smi)  # list of atomic tokens
        tokens = tf.compress(base, tokenizer.replace_root)
        token_count = len(tokens)
        
        # Collect data for all metrics
        total_tokens += token_count
        total_chars += len(smi)
        token_counts.append(token_count)
        token_freq.update(tokens)  # For entropy calculation
    
    n = len(token_counts)
    
    # Calculate basic metrics
    fertility = total_tokens / total_chars if total_chars > 0 else 0
    trie_avg = total_tokens / n if n > 0 else 0
    trie_var = sum((count - trie_avg) ** 2 for count in token_counts) / n if n > 0 else 0
    
    # Calculate normalized entropy
    if total_tokens == 0:
        normalized_entropy = 0.0
    else:
        vocab_size = len(token_freq)
        if vocab_size <= 1:
            normalized_entropy = 0.0
        else:
            # Shannon entropy
            entropy = -sum((cnt/total_tokens) * math.log2(cnt/total_tokens)
                          for cnt in token_freq.values())
            # Normalize by log₂(observed_vocab_size)
            normalized_entropy = entropy / math.log2(vocab_size)
    
    return fertility, trie_avg, trie_var, normalized_entropy

In [12]:
def _make_out_name(k: int, freq: int, ent: float) -> str:
    """
    Produce a file name that encodes the hyper-parameters, e.g.
    ttg_pubchem100K_K8_F4_H2p0.pkl
    (the dot in entropy is replaced by “p” to keep the name shell-safe).
    """
    ent_str = str(ent).replace(".", "p")
    return f"{OUT_DIR}/ttg_chebi_K{k}_F{freq}_H{ent_str}.pkl"

def run_ttg(k: int = 8, freq_thr: int = 4, entropy_thr: float = 2.0) -> str:
    """
    Build a TTG-guided compressor with the given hyper-parameters.
    Returns the path of the saved pickle.

    Example
    -------
    >>> run_ttg(k=10, freq_thr=3, entropy_thr=1.5)
    'ttg_chebi_K10_F3_H1p5.pkl'
    """
    out_path = _make_out_name(k, freq_thr, entropy_thr)

    print(f"Building TTG-guided trie compressor (K={k}, FREQ_THR={freq_thr}, "
          f"ENTROPY_THR={entropy_thr}) …")
    t0 = time.time()

    state = tf.prepare_compressor_with_ttg(
        iter_smiles(SLICE),
        K=k,
        freq_thr=freq_thr,
        entropy_thr=entropy_thr,
    )

    tf.save_state(state, out_path)
    print(f"✔ Trie saved → {out_path}  ({time.time() - t0:.1f}s)")

    #trie_fert, trie_avg, trie_var, trie_ent = compute_trie_metrics(SLICE, state)

    #print(f"✔ Trie Metrics Computed → {out_path}  ({time.time() - t0:.1f}s)")

    return out_path#, trie_fert, trie_avg, trie_var, trie_ent

In [13]:
run_ttg()

Building TTG-guided trie compressor (K=8, FREQ_THR=4, ENTROPY_THR=2.0) …
✔ Trie saved → ttg_vocab/ttg_chebi_K8_F4_H2p0.pkl  (170.5s)


'ttg_vocab/ttg_chebi_K8_F4_H2p0.pkl'

In [2]:
import time
from utils import iter_smiles
import trie_funcs as tf


In [ ]:
SLICE2 = "data/chebi_smiles.parquet"


In [ ]:
def run_ttg(k: int = 10, freq_thr: int = 3, entropy_thr: float = 3.5) -> str:
    """
    Build a TTG-guided compressor with the given hyper-parameters.
    Returns the path of the saved pickle.

    """

    print(f"Building TTG-guided trie compressor (K={k}, FREQ_THR={freq_thr}, "
          f"ENTROPY_THR={entropy_thr}) …")
    t0 = time.time()

    state = tf.prepare_compressor_with_ttg(
        iter_smiles(SLICE2),
        K=k,
        freq_thr=freq_thr,
        entropy_thr=entropy_thr,
    )

    tf.save_state(state, "ttg_chebi.pkl")
    print(f"✔ Trie saved →  ({time.time() - t0:.1f}s)")


In [4]:
run_ttg()

Building TTG-guided trie compressor (K=10, FREQ_THR=3, ENTROPY_THR=3.5) …
✔ Trie saved →  (75.1s)
